# P2 · Fleet Maintenance Planning

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from scipy.stats import ks_2samp
from sklearn.ensemble import HistGradientBoostingClassifier, HistGradientBoostingRegressor
from sklearn.model_selection import StratifiedKFold, train_test_split

SEED = 42
INSPECTION, SHORTAGE, SURPLUS = 10.0, 0.20, 0.02
SENSORS = [f"s{i:03d}" for i in range(170)]
SCHEMA = ["id", "flag", "budget"]

def find_data():
    required = {"train.csv", "test.csv", "sample_submission.csv"}
    kaggle = Path("/kaggle/input")
    candidates = [Path.cwd(), *(p.parent for p in sorted(kaggle.rglob("train.csv")))]
    matches = []
    for folder in dict.fromkeys(candidates):
        if all((folder / name).is_file() for name in required):
            test_cols = set(pd.read_csv(folder / "test.csv", nrows=0).columns)
            train_cols = set(pd.read_csv(folder / "train.csv", nrows=0).columns)
            if {"id", *SENSORS} <= test_cols and {"failure", "repair_cost", *SENSORS} <= train_cols:
                matches.append(folder)
    if len(matches) != 1:
        raise ValueError(f"Expected one P2 dataset. Matching directories: {matches}")
    return matches[0]

data = find_data()
train = pd.read_csv(data / "train.csv", dtype={"id": str})
test = pd.read_csv(data / "test.csv", dtype={"id": str})
sample = pd.read_csv(data / "sample_submission.csv", dtype={"id": str})
assert len(train) == 56_000 and len(test) == len(sample) == 20_000
assert list(sample) == SCHEMA
assert test.id.is_unique and sample.id.is_unique and set(test.id) == set(sample.id)
assert train.failure.isin([0, 1]).all()
failure = train.failure.to_numpy(dtype=int)
repair = train.repair_cost.fillna(0).to_numpy()
assert np.isfinite(repair).all() and np.array_equal(repair > 0, failure == 1)
print(f"{len(train):,} training vehicles, {failure.sum():,} recorded failures")

## Sensor readings

The test channels `s055`–`s063` match training channels `s054`–`s062` after a one-column shift. Scale them to a common resolution. Use stable counters to measure each reading relative to the vehicle’s overall sensor level.

In [ ]:
shift = {}
for name in SENSORS:
    old, new = train[name].dropna(), test[name].dropna()
    if min(old.nunique(), new.nunique()) < 2:
        shift[name] = 1.0
    else:
        distribution_shift = ks_2samp(old, new, method="asymp").statistic
        missingness_shift = abs(len(old) / len(train) - len(new) / len(test))
        shift[name] = max(distribution_shift, missingness_shift)

stable = [name for name in SENSORS if shift[name] < 0.10]

frequently_missing = train[stable].isna().mean().to_numpy() >= 0.20

def features(frame):
    readings = frame[stable].to_numpy(dtype="float32")
    missing = np.isnan(readings)
    counts = missing.sum(axis=1, dtype="float32")
    group_counts = missing[:, frequently_missing].sum(axis=1, dtype="float32")
    return np.column_stack((readings, counts, group_counts))

X, X_test = features(train), features(test)
print(f"Stable raw sensors: {len(stable)}")

# Normalize each reading by its training-depot scale before computing ratios.
scale = train[stable].quantile(0.90).to_numpy()
scale = np.where(scale > 0, scale, 1.0)
anchor_names = ["s071", "s123", "s149", "s053", "s012"]
anchor_cols = [stable.index(c) for c in anchor_names if c in stable]

# Adjacent columns match across depots after a one-column offset.
source_channels = [f"s{i:03d}" for i in range(54, 63)]
target_channels = [f"s{i:03d}" for i in range(55, 64)]
resolution = np.array([3, 10, 10, 100, 100, 100, 10, 10, 10], dtype="float32")
channel_scale = np.maximum(train[source_channels].quantile(0.9).to_numpy(dtype="float32"), 1)

def relative_features(frame, channels):
    raw = frame[stable].to_numpy(dtype="float32")
    scaled = raw / scale.astype("float32")
    exposure = pd.DataFrame(scaled[:, anchor_cols]).median(axis=1).to_numpy()
    log_readings = np.log1p(scaled)
    ratios = np.log1p(scaled / (0.01 + exposure[:, None]))
    parts = [log_readings, ratios, features(frame)[:, -2:]]
    # These blocks have the same missingness pattern and adjacent channels.
    for start, end in [(23, 33), (37, 47), (80, 90), (109, 119), (129, 139), (154, 164)]:
        cols = [stable.index(f"s{i:03d}") for i in range(start, end) if f"s{i:03d}" in stable]
        if len(cols) < 2:
            continue
        block = scaled[:, cols]
        total = np.sum(block, axis=1)
        parts.append(block / (0.01 + total[:, None]))
    block = frame[channels].to_numpy(dtype="float32")
    block = np.rint(block / resolution) * resolution / channel_scale
    parts.extend([np.log1p(block), np.log1p(block / (0.01 + exposure[:, None]))])
    return np.column_stack(parts).astype("float32")

Z, Z_test = relative_features(train, source_channels), relative_features(test, target_channels)

def active_channel_ratio(frame, name):
    active = frame[name].gt(100) & frame["s149"].gt(0)
    return frame.loc[active, name] / (1.0 + frame.loc[active, "s149"])

aligned_shift = [
    ks_2samp(active_channel_ratio(train, old), active_channel_ratio(test, new), method="asymp").statistic
    for old, new in zip(source_channels, target_channels)
]
print(f"Restored {len(aligned_shift)} channels. Largest active-ratio KS distance: {max(aligned_shift):.3f}")

## Failure and repair models

Boosted trees and a radial-basis kernel estimate failure risk. Two sparse regressions use the fractions and cumulative fractions within each sensor block to estimate log repair cost.

In [ ]:
from sklearn.linear_model import ElasticNet, LogisticRegression
from sklearn.preprocessing import StandardScaler

def fit_predict(fit_idx, query, seed):
    x, y, c = X[fit_idx], failure[fit_idx], repair[fit_idx]
    positive = y == 1
    cls_options = dict(learning_rate=0.075, max_iter=210, max_bins=127,
                       early_stopping=False, random_state=seed)
    probabilities = []
    for leaves, min_leaf in ((7, 25), (15, 40)):
        clf = HistGradientBoostingClassifier(
            **cls_options, max_leaf_nodes=leaves, min_samples_leaf=min_leaf,
            l2_regularization=3.0, max_features=0.85)
        clf.fit(x, y)
        probabilities.append(clf.predict_proba(query)[:, 1])
    p = np.mean(probabilities, axis=0)

    mean_cost = c[positive].mean()
    weights = np.where(positive, c / mean_cost, 1.0)
    weighted = HistGradientBoostingClassifier(
        **cls_options, max_leaf_nodes=7, min_samples_leaf=35,
        l2_regularization=5.0, max_features=0.85)
    weighted.fit(x, y, sample_weight=weights)
    p_weighted = np.clip(weighted.predict_proba(query)[:, 1], 0, 0.985)
    expected_cost = mean_cost * p_weighted / (1 - p_weighted) * (1 - p)

    severity = HistGradientBoostingRegressor(
        max_leaf_nodes=5, min_samples_leaf=20, max_iter=65,
        learning_rate=0.06, l2_regularization=4.0, early_stopping=False,
        random_state=seed)
    severity.fit(x[positive], np.log(c[positive]))
    return np.column_stack((p, expected_cost, severity.predict(query)))


def relative_predictions(indices, query, seed):
    x, y, c = Z[indices], failure[indices], repair[indices]
    positive = y == 1
    boost = HistGradientBoostingClassifier(max_leaf_nodes=12, min_samples_leaf=30, max_iter=175,
                                           max_bins=127, learning_rate=0.065, max_features=0.7,
                                           l2_regularization=6, early_stopping=False, random_state=seed)
    boost.fit(x, y, sample_weight=np.where(positive, 6.0, 1.0))
    score = boost.predict_proba(query)[:, 1]
    p_boost = score / (6.0 - 5.0 * score)

    # Sparse log-cost regression on the failed vehicles.
    def severity_features(rows):
        return np.column_stack((np.nan_to_num(rows, nan=0), np.isnan(rows), rows > 0))
    scaler = StandardScaler()
    design = scaler.fit_transform(severity_features(x[positive]))
    model = ElasticNet(alpha=0.09, l1_ratio=0.55, max_iter=4000)
    model.fit(design, np.log(c[positive]))
    log_cost = model.predict(scaler.transform(severity_features(query)))
    log_cost = np.clip(log_cost, np.log(c[positive].min()), np.log(c.max()))
    return np.column_stack((p_boost, log_cost))



from sklearn.kernel_approximation import Nystroem

def kernel_risk(indices, query, seed):
    # Fit the distance metric on the part of the training fleet with substantial sensor activity.
    x = Z[indices]
    y = failure[indices]
    active = x[:, stable.index("s071")]
    high = active >= np.nanquantile(active, 0.90)
    support = high | (y == 1)
    pool = np.nan_to_num(x[support], nan=-0.75)
    median = np.median(pool,axis=0)
    lo, hi = np.quantile(pool,[0.15,0.85],axis=0)
    spread = np.maximum(hi-lo,0.15)
    def standardized(raw):
        return np.clip((np.nan_to_num(raw,nan=-0.75) - median)/spread,-4,4)
    balanced = standardized(pool)
    # Prefer channels separating recorded failures from other high-activity vehicles.
    positive = y[support] == 1
    a, b = balanced[positive], balanced[~positive]
    separation = (a.mean(axis=0)-b.mean(axis=0))**2 / (a.var(axis=0)+b.var(axis=0)+0.02)
    select = np.argsort(separation)[-120:]
    def inputs(raw):
        return standardized(raw)[:,select]
    rng=np.random.default_rng(seed)
    difficult=np.flatnonzero(high & (y==0))
    landmarks=np.r_[np.flatnonzero(y==1),rng.choice(difficult,min(480,len(difficult)),replace=False)]
    # A low-rank RBF kernel keeps scoring all vehicles inexpensive.
    mapper=Nystroem(kernel="rbf",gamma=0.011,n_components=350,random_state=seed)
    mapper.fit(inputs(x[landmarks]))
    def transform(raw):
        return np.vstack([mapper.transform(inputs(raw[i:i+2500])) for i in range(0,len(raw),2500)])
    low=np.flatnonzero(~high & (y==0))
    sampled=rng.choice(low,min(len(low),6500),replace=False)
    rows=np.r_[np.flatnonzero(support),sampled]
    weights=np.where(y[rows]==1,8.0,np.where(high[rows],1.0,len(low)/len(sampled)))
    model=LogisticRegression(C=12,max_iter=350)
    model.fit(transform(x[rows]),y[rows],sample_weight=weights)
    s=model.predict_proba(transform(query))[:,1]
    return s/(8-7*s)

def channel_shapes(raw):
    parts = []
    # Each contiguous block is a set of readings with a shared acquisition pattern.
    for lo, hi in [(23,33), (37,47), (54,63), (80,90), (109,119), (129,139), (154,164)]:
        cols = [(stable + source_channels).index(f"s{i:03d}") for i in range(lo,hi)]
        block = raw[:,cols]
        total = block.sum(axis=1,keepdims=True)
        share = block / np.maximum(total,1.)
        left = np.cumsum(share,axis=1)[:,:-1]
        right = np.cumsum(share[:,::-1],axis=1)[:,-2::-1]
        ordinal = np.arange(len(cols)) / (len(cols)-1)
        mean = share @ ordinal
        variance = np.maximum(0,share @ (ordinal**2)-mean**2)
        summary = np.column_stack((mean, np.sqrt(variance), (share**2).sum(axis=1), share.max(axis=1)))
        parts.extend([summary,left,np.log1p(1000*left),np.log1p(1000*right)])
    return np.column_stack(parts)

# Read channel magnitudes before normalizing by the fleet-wide 90th percentile.
def cost_readings(frame, channels):
    raw = frame[stable].to_numpy(dtype=np.float64)
    matched = frame[channels].to_numpy(dtype=np.float64)
    return np.column_stack((raw, matched))

def cost_basis(raw, positive_scale):
    # The first transform retains low positive readings in heavy-tailed channels.
    logged = np.log1p(raw)
    relative = raw / positive_scale
    parts = [logged, np.log1p(10 * relative), np.arcsinh(relative), np.sqrt(np.clip(relative,0,25))]
    for lo, hi in [(23,33), (37,47), (54,63), (80,90), (109,119), (129,139), (154,164)]:
        cols = [(stable + source_channels).index(f"s{i:03d}") for i in range(lo,hi)]
        block = raw[:,cols]
        total = block.sum(axis=1, keepdims=True)
        share = block / np.maximum(total, 1.)
        parts.extend([share, np.sqrt(share), np.log1p(100*share)])
    return np.column_stack(parts)

def fit_cost_models(raw_fit, z_fit, costs, raw_query, z_query):
    positive_scale = np.nanquantile(raw_fit, 0.75, axis=0)
    positive_scale = np.where(np.isfinite(positive_scale) & (positive_scale > 1), positive_scale, 1)
    def design(raw, z):
        parts = [z, np.isnan(z), z > 0, cost_basis(raw, positive_scale), channel_shapes(raw)]
        return np.nan_to_num(np.column_stack(parts), nan=0).astype(np.float64)

    scaler = StandardScaler()
    x = scaler.fit_transform(design(raw_fit, z_fit))
    log_cost = np.log(costs)
    first = ElasticNet(alpha=0.075, l1_ratio=0.8, max_iter=5000).fit(x, log_cost)
    strength = np.abs(first.coef_)
    penalty = np.maximum(np.sqrt(strength / max(strength.max(), 1e-12)), 0.09)
    second = ElasticNet(alpha=0.035, l1_ratio=0.75, max_iter=5000).fit(x * penalty, log_cost)

    # Both regressions use the same sensor transformations.
    coefficients = np.column_stack((first.coef_, second.coef_ * penalty))
    intercepts = np.array([first.intercept_, second.intercept_])
    predictions = []
    for start in range(0, len(raw_query), 2500):
        batch = slice(start, start + 2500)
        values = scaler.transform(design(raw_query[batch], z_query[batch]))
        predictions.append(values @ coefficients + intercepts)
    return np.clip(np.vstack(predictions), np.log(costs.min()), np.log(costs.max()))

## Inspection and parts budget

Calibrate failure risk on held-out predictions and adjust it for the estimated new-depot failure rate. Budget at the 10/11 quantile of parts demand, including zero demand on healthy vehicles.

In [ ]:
def estimate_prior(indices):
    """Fit a two-class mixture to high-risk tails of stable sensors."""
    old = X[indices, :len(stable)]
    new = X_test[:, :len(stable)]
    y = failure[indices].astype(bool)
    candidates = []
    for col, name in enumerate(stable):
        if shift[name] >= 0.06:
            continue
        positive, negative = old[y, col], old[~y, col]
        if np.isfinite(positive).sum() < 0.85 * y.sum():
            continue
        direction = 1 if np.nanmedian(positive) >= np.nanmedian(negative) else -1
        cutoffs = np.nanquantile(direction * negative, [0.95, 0.97, 0.98, 0.99])
        positive_tail = (direction * positive[:, None] > cutoffs).mean(axis=0)
        negative_tail = (direction * negative[:, None] > cutoffs).mean(axis=0)
        test_tail = (direction * new[:, col, None] > cutoffs).mean(axis=0)
        contrast = positive_tail - negative_tail
        if contrast[0] >= 0.55:
            estimate = np.median((test_tail - negative_tail) / np.maximum(contrast, 1e-6))
            candidates.append((contrast[0], name, estimate))
    # Use the strongest class separators and a median to limit the effect of shifted sensors.
    top = sorted(candidates, reverse=True)[:12]
    if len(top) < 3:
        return y.mean(), top
    return float(np.clip(np.median([estimate for _, _, estimate in top]), 0.002, 0.15)), top


def prior_ratio(source, target):
    return (target / (1 - target)) / (source / (1 - source))


def vehicle_weights(actual, target, source):
    return np.where(actual > 0, target / source, (1 - target) / (1 - source))


def individual_cost(actual, flag, budget):
    return np.where(flag, INSPECTION + SHORTAGE * np.maximum(actual - budget, 0)
                    + SURPLUS * np.maximum(budget - actual, 0), actual)




def calibrated_demand(pred, reference, actual, target_prior):
    observed = actual > 0
    def log_odds(p):
        return logit(np.clip(p, 1e-7, 1 - 1e-7)).reshape(-1, 1)

    calibration = LogisticRegression(C=1)
    calibration.fit(log_odds(reference[:, 0]), observed)
    source_p = calibration.predict_proba(log_odds(pred[:, 0]))[:, 1]

    # Calibrate the spread of log-cost predictions using only held-out failures.
    y = np.log(actual[observed])
    # Learn a convex blend from held-out log costs. Shrink its weight toward equal shares.
    tree = reference[observed, 2]
    difference = reference[observed, 3] - tree
    weight = np.clip((difference @ (y - tree) + 5) / (difference @ difference + 10), 0, 1)
    x = tree + weight * difference
    severity = pred[:, 2] + weight * (pred[:, 3] - pred[:, 2])
    slope = np.clip((np.dot(x - x.mean(), y - y.mean()) + 20) /
                    (np.dot(x - x.mean(), x - x.mean()) + 20), 0.5, 1.75)
    center = np.exp(np.clip(y.mean() + slope * (severity - x.mean()), -4, 11))
    residuals = np.sort(np.exp(y - y.mean() - slope * (x - x.mean())))
    prefix = np.r_[0, np.cumsum(residuals)]
    ceiling = actual.max()
    limit = np.searchsorted(residuals, ceiling / center, side="right")
    mean_cost = (center * prefix[limit] + ceiling * (len(residuals) - limit)) / len(residuals)
    odds = prior_ratio(observed.mean(), target_prior)
    return source_p, odds, center, residuals, prefix, ceiling, mean_cost


def calibrated_policy(demand, expected_raw, multiplier):
    source_p, odds, center, residuals, prefix, ceiling, mean_cost = demand
    # The cost-weighted classifier supplies a second estimate of the skip loss.
    # Keep the budget tied to the calibrated demand distribution.
    adjusted_odds = odds * multiplier
    p = adjusted_odds * source_p / (1 + (adjusted_odds - 1) * source_p)
    rate = SURPLUS / (SHORTAGE + SURPLUS)
    q = 1 - rate / np.maximum(p, 1e-12)
    ix = np.clip(np.ceil(q * len(residuals)).astype(int) - 1, 0, len(residuals) - 1)
    budget = np.where(q > 0, np.minimum(center * residuals[ix], ceiling), 0.0)
    below = np.searchsorted(residuals, budget / center, side="right")
    positive_excess = np.maximum((budget * below - center * prefix[below]) / len(residuals), 0)
    shortage = np.maximum(mean_cost - budget + positive_excess, 0)
    inspect = INSPECTION + SHORTAGE * p * shortage + SURPLUS * ((1-p)*budget + p*positive_excess)
    direct = expected_raw * p / np.maximum(source_p, 1e-7)
    skip = 0.8 * p * mean_cost + 0.2 * direct
    flag = skip > inspect
    return flag.astype(int), np.where(flag, np.round(budget, 2), 0.0)



from scipy.special import expit, logit
from sklearn.linear_model import LogisticRegression

def risk_inputs(pred):
    # Each column is a held-out failure score. Combine the scores on the log-odds scale.
    scores = np.clip(pred[["raw_risk", "relative_risk", "kernel_risk"]].to_numpy(), 1e-6, 1 - 1e-6)
    logits = logit(scores)
    average = logits[:, [0, 1]].mean(axis=1)
    return np.column_stack((logits, np.maximum(average + 4.5, 0), np.maximum(average + 1.5, 0)))


def calibrated_risk(ref, query, actual, target_rate):
    a, b = risk_inputs(ref), risk_inputs(query)
    obs = actual > 0
    model = LogisticRegression(C=0.5, max_iter=300)
    model.fit(a, obs)
    recorded = model.predict_proba(b)[:, 1]
    recorded_target = expit(logit(recorded) + np.log(prior_ratio(obs.mean(), target_rate)))
    return recorded_target


def shape_demand(ref, query, actual):
    observed = actual > 0
    y = np.log(actual[observed])
    def severity(pred):
        return pred[["sparse_cost", "shape_cost", "adaptive_cost"]].to_numpy() @ [0.15, 0.425, 0.425]
    x = severity(ref[observed])
    slope = np.clip(((x-x.mean()) @ (y-y.mean()) + 12) / ((x-x.mean()) @ (x-x.mean()) + 12),.65,1.6)
    center = y.mean() + slope*(severity(query)-x.mean())
    errors = y - y.mean() - slope*(x-x.mean())
    return np.exp(np.clip(center,-4,11)), np.sort(np.exp(errors)), actual.max()


def parts_decisions(p, demand, expected_raw, raw_to_target, direct_share):
    scale, residuals, ceiling = demand
    prefix = np.r_[0,np.cumsum(residuals)]
    n = len(residuals)
    cap = np.searchsorted(residuals, ceiling / scale, side="right")
    mean = (scale * prefix[cap] + (n - cap)*ceiling) / n
    q = 1 - SURPLUS / ((SHORTAGE + SURPLUS) * np.maximum(p, 1e-12))
    index = np.clip(np.ceil(n*q).astype(int)-1, 0, n-1)
    budget = np.where(q>0,np.minimum(scale*residuals[index],ceiling),0.)
    below = np.searchsorted(residuals, budget/scale, side="right")
    returned = np.maximum((budget*below - scale*prefix[below])/n,0)
    shortage = np.maximum(mean - budget + returned,0)
    inspect = INSPECTION + SHORTAGE*p*shortage + SURPLUS*((1-p)*budget + p*returned)
    expected = p*mean
    direct = np.clip(raw_to_target*expected_raw, expected/3, 3*expected)
    flag = (1-direct_share)*expected + direct_share*direct > inspect
    return flag.astype(int), np.where(flag,np.round(budget,2),0)


def decide(reference, query, actual, prior):
    def previous(frame):
        return np.column_stack((
            0.45*frame.raw_risk + 0.55*frame.relative_risk, frame.direct_cost,
            frame.tree_cost, frame.sparse_cost))
    base = calibrated_demand(previous(query), previous(reference), actual, prior)
    old = calibrated_policy(base, query.direct_cost.to_numpy(), 1.0)
    base_target = expit(logit(base[0]) + np.log(base[1]))
    stacked = calibrated_risk(reference, query, actual, prior)
    risk = 0.7 * stacked + 0.3 * base_target
    risk = expit(logit(risk) + np.log(1.15))
    demand = shape_demand(reference, query, actual)
    odds_correction = risk / np.maximum(base[0], 1e-7)
    new = parts_decisions(risk, demand, query.direct_cost.to_numpy(), odds_correction, 0.15)
    return old, new

## Five-fold cost comparison

Fit each model on four folds and predict the fifth. Compare the new decisions with the previous policy at three assumed new-depot failure rates. Each fold’s calibration uses the other folds’ out-of-fold predictions.

In [ ]:
raw_cost = cost_readings(train, source_channels)
test_cost = cost_readings(test, target_channels)
names = ["raw_risk", "direct_cost", "tree_cost", "relative_risk", "sparse_cost",
         "kernel_risk", "shape_cost", "adaptive_cost"]
indices = np.arange(len(train))
folds = list(StratifiedKFold(5, shuffle=True, random_state=SEED+1).split(indices, failure))
oof = pd.DataFrame(np.nan, index=indices, columns=names)
test_folds = []
for fold, (fit, val) in enumerate(folds, 1):
    seed = SEED + 1 + fold
    pos = fit[failure[fit] == 1]
    z_query = np.vstack((Z[val], Z_test))
    raw_query = np.vstack((raw_cost[val], test_cost))
    predictions = np.column_stack((
        fit_predict(fit, np.vstack((X[val], X_test)), seed),
        relative_predictions(fit, z_query, seed),
        kernel_risk(fit, z_query, SEED+8+fold),
        fit_cost_models(raw_cost[pos], Z[pos], repair[pos], raw_query, z_query),
    ))
    oof.iloc[val] = predictions[:len(val)]
    test_folds.append(predictions[len(val):])
    print(f"Finished fold {fold}/{len(folds)}", flush=True)
test_scores = pd.DataFrame(np.mean(test_folds,axis=0), columns=names)
assert np.isfinite(oof.to_numpy()).all() and np.isfinite(test_scores.to_numpy()).all()

prior, _ = estimate_prior(indices)
losses = np.empty((len(train), 2))
for fit, val in folds:
    fold_prior, _ = estimate_prior(fit)
    policies = decide(oof.iloc[fit], oof.iloc[val], repair[fit], fold_prior)
    for j, (flag, budget) in enumerate(policies):
        losses[val, j] = individual_cost(repair[val], flag, budget)
rows = []
for target_rate in prior * np.array([0.65, 1.0, 1.35]):
    weights = vehicle_weights(repair, target_rate, failure.mean())
    rows.append([target_rate, *np.average(losses, axis=0, weights=weights)])
print(pd.DataFrame(rows, columns=["assumed rate", "previous cost/vehicle", "new cost/vehicle"]).round(4).to_string(index=False))

## Kaggle submission

Average the five fold predictions and write `/kaggle/working/submission.csv`.

In [ ]:
_, (flag, budget) = decide(oof, test_scores, repair, prior)
predictions = pd.DataFrame({"id": test.id, "flag": flag, "budget": budget}).set_index("id")
submission = predictions.loc[sample.id].reset_index()

output_dir = Path("/kaggle/working")
if not output_dir.is_dir():
    output_dir = Path.cwd()
output = output_dir / "submission.csv"
submission.to_csv(output, index=False)
written = pd.read_csv(output, dtype={"id": str})
assert written.columns.tolist() == SCHEMA
assert len(written) == 20_000 and written.id.is_unique and written.id.equals(sample.id)
assert written.flag.isin([0, 1]).all()
assert np.isfinite(written.budget).all() and written.budget.ge(0).all()
assert written.loc[written.flag.eq(0), "budget"].eq(0).all()
print(f"Saved {len(written):,} rows to {output}")
print(f"Inspections: {written.flag.sum():,}. Positive budgets: {written.budget.gt(0).sum():,}.")
print(written.head().to_string(index=False))